# Investigating the simplification of geometry
**Author**:  Sian Teesdale 

**Date created**:  14/04/2026

**Dataset Scope**: [one/many/all datasets]  

**Purpose**: The aim is to see what part of our transformation pipeline is responsible for simplifying geometries before they appear on the Planning platform.

In [ ]:
import requests
import tempfile
import os

import pandas as pd
import geopandas as gpd

In [4]:
# Read in conservation-area
platform_data = gpd.read_file("https://files.planning.data.gov.uk/dataset/conservation-area.geojson")

# Read in LPA boundary
boundary = gpd.read_file("https://files.planning.data.gov.uk/dataset/local-planning-authority.geojson")


This investigation came from Barnet, who highlighted they're seeing a difference in their raw data (endpoint) and our data. I've spotted this entity, through QGIS, that has the slightest variations that would be good to test.

- entity: 44002422
- entry-date: 20/03/2025 00:00:00 (GMT)
- name: Totteridge
- organisation-entity: 48
- prefix: conservation-area
- quality: authoritative
- reference: CA12
- start-date: 2008-05-06
- document-url: https://open.barnet.gov.uk/download/2nx73/evh
- documentation-url: https://www.barnet.gov.uk/node/9022


In [ ]:
# Read in Barnet's data by bypassing their server block
url = "https://open.barnet.gov.uk/download/20yo8/c6n/conservation_area.gpkg"
headers = {"User-Agent": "Mozilla/5.0"}

with tempfile.NamedTemporaryFile(suffix=".gpkg", delete=False) as f:
    f.write(requests.get(url, headers=headers).content)
    tmp_path = f.name

barnet_data = gpd.read_file(tmp_path)
os.unlink(tmp_path)


In [ ]:
# Save Totteridge example
platform_totteridge = platform_data[platform_data["name"] == "Totteridge"]
barnet_totteridge = barnet_data[barnet_data["name"] == "Totteridge"]

In [64]:
platform_data.loc[platform_data['name'] == 'Wood Street']

,dataset,end-date,entity,entry-date,name,organisation-entity,prefix,quality,reference,start-date,typology,document-url,legislation,notes,documentation-url,designation-date,geometry
1690,conservation-area,,44002423,2025-03-20,Wood Street,48,conservation-area,authoritative,CA03,2007-06-07,geography,https://open.barnet.gov.uk/download/2nx73/pph,None,None,https://www.barnet.gov.uk/node/9023,1969-04-25,"POLYGON ((-0.20169 51.65448, -0.20174 51.65478..."
3044,conservation-area,,44004270,2026-04-03,Wood Street,151,conservation-area,some,8669,1973-12-18,geography,None,None,No Appraisal document.,https://www.guildford.gov.uk/newlocalplan/CHtt...,NaT,"POLYGON ((-0.6354 51.25177, -0.63543 51.25176,..."


---
First let's explore the data and see if the geometries are matching currently

In [19]:
# Check Barnet uses the same CRS as the platform
print("Platform CRS:", platform_totteridge.crs, "; Barnet CRS:", barnet_totteridge.crs)
barnet_totteridge.crs == platform_totteridge.crs

Platform CRS: EPSG:4326 ; Barnet CRS: EPSG:4326


True

In [12]:
# Exact equality (same coordinates, same order)
platform_totteridge.geometry.iloc[0].equals(barnet_totteridge.geometry.iloc[0])

False

In [13]:
# With tolerance for floating-point differences
platform_totteridge.geometry.iloc[0].equals_exact(barnet_totteridge.geometry.iloc[0], tolerance=1e-6)

False

This confirms the platform and raw data is not matching for Totteridge.


---

The geometry goes through several transformation stages, all laid out in the wkt.py script (taken from https://github.com/digital-land/digital-land-python/blob/8974e00c083a5247e8dbd7b663af27e0f26c8a7e/digital_land/datatype/wkt.py)

- Conversion to WKT
- Transforming of CRS to WGS84
- Degrees of precision, we use 6 d.p. for coordinates as a standard
- Transformations of geometry from polygons/geometry collection to multipolygons
- Simplifications/normalisation of the geometry (which can form part of a cyclic cycle with the previous point)

We will now go through each stage to test which stage is changing the geometry

In [20]:
import shapely.wkt
from shapely import set_precision
from shapely.geometry import MultiPolygon
from shapely.geometry.polygon import orient
from shapely.validation import explain_validity, make_valid
from shapely.ops import transform
from pyproj import Transformer
from pyproj.transformer import TransformerGroup

# Copy of the standalone functions from wkt.py (no DataType dependency)
osgb_to_wgs84 = Transformer.from_crs(27700, 4326, always_xy=True)

def dump_wkt(geometry, precision=6, dimensions=2):
    wkt = shapely.wkt.dumps(geometry, rounding_precision=precision, output_dimension=dimensions)
    return wkt.replace(", ", ",")

def make_multipolygon(geometry):
    if geometry.geom_type == "MultiPolygon":
        return geometry
    if geometry.geom_type == "Polygon":
        return MultiPolygon([geometry])
    if geometry.geom_type == "GeometryCollection":
        polygons = []
        for geom in geometry.geoms:
            if geom.geom_type == "Polygon":
                polygons.append(geom)
            elif geom.geom_type == "MultiPolygon":
                polygons.extend(geom.geoms)
        return MultiPolygon(polygons)
    return None


In [25]:
# --- Step-by-step pipeline ---
raw = barnet_totteridge.geometry.iloc[0]
print(f"Raw type: {raw.geom_type}")

# Step 1: CRS conversion (if OSGB → WGS84)
# barnet_data.crs tells us if this applies
print(f"barnet CRS: {barnet_data.crs}")
# If already EPSG:4326, skip; if EPSG:27700, apply:
if barnet_data.crs.to_epsg() == 27700:
    step1 = transform(osgb_to_wgs84.transform, raw)
    print(f"Step 1 (CRS conversion) changed geometry: {not raw.equals(step1)}")
else:
    step1 = raw
    print("Step 1 (CRS conversion): skipped, already WGS84")

# Step 2: dump_wkt + reload (reduces to 6 d.p.)
wkt_str = dump_wkt(step1)
step2 = shapely.wkt.loads(wkt_str)
print(f"Step 2 (6 d.p. precision round-trip) changed geometry: {not step1.equals(step2)}")
print(f"  equals_exact(1e-7): {step1.equals_exact(step2, 1e-7)}")

# Step 3: simplify(0.000005)
step3 = step2.simplify(0.000005)
print(f"Step 3 (simplify 0.000005) changed geometry: {not step2.equals(step3)}")

# Step 4: set_precision(0.000001, pointwise)
step4 = set_precision(step3, 0.000001, mode="pointwise")
print(f"Step 4 (set_precision 0.000001) changed geometry: {not step3.equals(step4)}")

# Step 5: make_valid (only if invalid)
if not step4.is_valid:
    step5 = make_valid(step4)
    print(f"Step 5 (make_valid) changed geometry: {not step4.equals(step5)} — was invalid: {explain_validity(step4)}")
else:
    step5 = step4
    print("Step 5 (make_valid): skipped, geometry is valid")

# Step 6: make_multipolygon
step6 = make_multipolygon(step5)
print(f"Step 6 (make_multipolygon) changed geometry: {not step5.equals(step6)}")
print(f"  type before: {step5.geom_type}, after: {step6.geom_type}")

# Step 6b: buffer(0) if invalid after make_multipolygon
if step6 and not step6.is_valid:
    step6b = step6.buffer(0)
    print(f"Step 6b (buffer(0)) changed geometry: {not step6.equals(step6b)}")
else:
    step6b = step6
    print("Step 6b (buffer(0)): skipped, geometry is valid")

# Step 6c: second make_multipolygon
step6c = make_multipolygon(step6b)
print(f"Step 6c (2nd make_multipolygon) changed geometry: {not step6b.equals(step6c)}")

# Step 7: orient (fix winding order)
polygons = [orient(g) for g in step6.geoms]
step7 = MultiPolygon(polygons)
print(f"Step 7 (orient/winding order) changed geometry: {not step6.equals(step7)}")

# Final comparison against platform_totteridge
final = step7
platform_geom = platform_totteridge.geometry.iloc[0]
print(f"\nFinal vs platform_totteridge equals: {final.equals(platform_geom)}")
print(f"Final vs platform_totteridge equals_exact(1e-6): {final.equals_exact(platform_geom, 1e-6)}")


Raw type: Polygon
barnet CRS: EPSG:4326
Step 1 (CRS conversion): skipped, already WGS84
Step 2 (6 d.p. precision round-trip) changed geometry: True
  equals_exact(1e-7): False
Step 3 (simplify 0.000005) changed geometry: True
Step 4 (set_precision 0.000001) changed geometry: True
Step 5 (make_valid): skipped, geometry is valid
Step 6 (make_multipolygon) changed geometry: False
  type before: Polygon, after: MultiPolygon
Step 6b (buffer(0)): skipped, geometry is valid
Step 6c (2nd make_multipolygon) changed geometry: False
Step 7 (orient/winding order) changed geometry: False

Final vs platform_totteridge equals: False
Final vs platform_totteridge equals_exact(1e-6): False


The results suggest a few things:
- Step 2 — 6 d.p. round-trip: The raw Barnet data has more than 6 decimal places of coordinate precision. Dumping to WKT and reloading rounds every coordinate, introducing small but measurable changes to every vertex. This is the first lossy operation.

- Step 3 — simplify(0.000005): Vertices are removed if they fall within ~0.5m of the simplified line. This reduces the number of points in the geometry, meaningfully changing its shape.

- Step 4 — set_precision(0.000001): Coordinates are snapped to a 1m grid, introducing further rounding on top of steps 2 and 3.

- Steps 6 and 7 are lossless — wrapping a Polygon into a MultiPolygon and fixing winding order don't change the underlying coordinates.

The final result still doesn't match platform_totteridge, which means either:
- The platform data was processed from a different vintage of the Barnet source file
- There are additional transformations upstream not captured in wkt.py (e.g. in parse_wkt, or earlier in the pipeline before this code runs)

SummaryL the pipeline does change geometries, with the precision round-trip, simplification, and set_precision being the three culprits. The cumulative effect is that the final geometry is a simplified, lower-precision approximation of the original.

---

Checking the platform_torridge data matches what's in datasette

In [26]:
entity = '44002422'

In [27]:
# Read in json from API
api_url = 'https://datasette.planning.data.gov.uk/conservation-area/fact.json?_sort=fact&entity__exact=44002422&_labels=on'
response = requests.get(api_url)
if response.status_code == 200:
    data = response.json()
    print("API response data:", data)
else:
    print(f"API request failed with status code {response.status_code}")

API response data: {'database': 'conservation-area', 'table': 'fact', 'is_view': False, 'human_description_en': 'where entity = 44002422 sorted by fact', 'rows': [{'end_date': '', 'entity': {'value': 44002422, 'label': 'Totteridge'}, 'fact': '064ed0e91aaa0867d53039784b4511d40101bbf9651722ad58e572dada1241ff', 'field': 'documentation-url', 'entry_date': '2024-02-08', 'priority': 2, 'reference_entity': '', 'start_date': '', 'value': 'https://open.barnet.gov.uk/download/2nx73/evh'}, {'end_date': '', 'entity': {'value': 44002422, 'label': 'Totteridge'}, 'fact': '09036ca39726a8eee47b362c2020ea7b278ed393f726bb496b79ddf604e1b35f', 'field': 'entry-date', 'entry_date': '2025-07-03', 'priority': 1, 'reference_entity': '', 'start_date': '', 'value': '2025-07-03'}, {'end_date': '', 'entity': {'value': 44002422, 'label': 'Totteridge'}, 'fact': '0bd4c57324c65b33235845c600313e273e1d5c5317d8d975722d059bd5b68ead', 'field': 'geometry', 'entry_date': '2023-07-22', 'priority': 1, 'reference_entity': '', 's

In [29]:
# Load the data into a DataFrame
datasette = pd.DataFrame(data['rows'], columns=data['columns'])

In [34]:
datasette_lpa = datasette.loc[(datasette['field'] == 'geometry') & (datasette['priority'] == 2)]

In [39]:
# Extract the two geometries from datasette_lpa
# 'value' is the last column (index 8)
datasette_geom_1 = shapely.wkt.loads(datasette_lpa.iloc[0, -1])  # entry_date: 2025-03-20
datasette_geom_2 = shapely.wkt.loads(datasette_lpa.iloc[1, -1])  # entry_date: 2024-06-05

platform_geom = platform_totteridge.geometry.iloc[0]

print("--- Row 1 (2025-03-20) vs platform ---")
print(f"  equals:            {datasette_geom_1.equals(platform_geom)}")
print(f"  equals_exact(1e-6):{datasette_geom_1.equals_exact(platform_geom, 1e-6)}")

print("\n--- Row 2 (2024-06-05) vs platform ---")
print(f"  equals:            {datasette_geom_2.equals(platform_geom)}")
print(f"  equals_exact(1e-6):{datasette_geom_2.equals_exact(platform_geom, 1e-6)}")

print("\n--- Row 1 vs Row 2 ---")
print(f"  equals:            {datasette_geom_1.equals(datasette_geom_2)}")
print(f"  equals_exact(1e-6):{datasette_geom_1.equals_exact(datasette_geom_2, 1e-6)}")


--- Row 1 (2025-03-20) vs platform ---
  equals:            True
  equals_exact(1e-6):False

--- Row 2 (2024-06-05) vs platform ---
  equals:            False
  equals_exact(1e-6):False

--- Row 1 vs Row 2 ---
  equals:            False
  equals_exact(1e-6):False


In [40]:
# Check if symmetric difference (i.e. area that differs) is zero
diff = datasette_geom_1.symmetric_difference(platform_geom)
print(f"Symmetric difference area: {diff.area}")  # expect 0.0 if truly identical

Symmetric difference area: 0.0


Good news - the correct data on Datasette is matching the Planning platform!

---

Now let's use the actual wkt.py file from Digital Land - I think my trying to simplify the steps above is resulting in more changes that expected

In [41]:
from digital_land.datatype.wkt import WktDataType

wkt_normaliser = WktDataType()

# Get raw WKT from barnet (as a string, which is what the pipeline receives)
raw_wkt = barnet_totteridge.geometry.iloc[0].wkt

# Run it through the full pipeline
result_wkt = wkt_normaliser.normalise(raw_wkt)

# Load result as geometry for comparison
result_geom = shapely.wkt.loads(result_wkt)
platform_geom = platform_totteridge.geometry.iloc[0]

print(f"Result equals platform:       {result_geom.equals(platform_geom)}")
print(f"Result equals_exact(1e-6):    {result_geom.equals_exact(platform_geom, 1e-6)}")
print(f"Symmetric difference area:    {result_geom.symmetric_difference(platform_geom).area}")


/Users/sianteesdale/Documents/GitHub/jupyter-analysis/.venv/lib/python3.10/site-packages/pyproj/transformer.py:207: UserWarning: Best transformation is not available due to missing Grid(short_name=uk_os_OSTN15_NTv2_OSGBtoETRS.tif, full_name=, package_name=, url=https://cdn.proj.org/uk_os_OSTN15_NTv2_OSGBtoETRS.tif, direct_download=True, open_license=True, available=False)
  super().__init__(


Result equals platform:       False
Result equals_exact(1e-6):    False
Symmetric difference area:    9.099999977025483e-11


The geometry difference is negligible. 9.1e-11 square degrees is approximately 0.7 mm² at UK latitudes. This is pure floating-point arithmetic noise, and not a meaningful geometric difference. This confirms the pipeline is producing the right geometry with the current code

In [47]:
print(f"Symmetric difference area (m²): {result_geom.symmetric_difference(platform_geom).area * 111000 * 70000:.6f}")
# ~0.7 mm² at UK latitudes

# Also check vertex counts match (same simplification occurred)
result_coords = len(list(result_geom.geoms[0].exterior.coords))
platform_coords = len(list(platform_geom.exterior.coords))  # Polygon, no .geoms
print(f"Result vertices:   {result_coords}")
print(f"Platform vertices: {platform_coords}")



Symmetric difference area (m²): 0.707070
Result vertices:   520
Platform vertices: 520


Both results confirm the pipeline is working correctly:

- 0.7 mm² symmetric difference — sub-pixel, floating-point noise only
- 520 vertices each — identical simplification was applied

---

Now we've confirmed the pipeline I'm running is the same as the platform data, let's see what step is actually producing the changes in geometry. My three culprits I want to check are:
1. 6 d.p.
2. Simplification
3. Precision

In [57]:
def report(name, geom, prev=None, ref=None):
    try:
        coords = len(list(geom.geoms[0].exterior.coords))
    except AttributeError:
        coords = len(list(geom.exterior.coords))
    area_diff = f"{geom.symmetric_difference(ref).area * 111000 * 70000:.4f} m²" if ref else "-"
    prev_diff = f"{geom.symmetric_difference(prev).area * 111000 * 70000:.4f} m²" if prev else "-"
    print(f"{name:<35} vertices={coords:>4}  vs_raw={area_diff}  vs_prev={prev_diff}")

def check_geometry(df, name):
    raw_df = df.loc[df['name'] == name]
    raw = raw_df.geometry.iloc[0]

    print(f"\n--- Checking geometry for '{name}' ---")

    report("0. Raw", raw)

    step2 = shapely.wkt.loads(dump_wkt(raw))
    report("1. After applying 6 d.p.", step2, raw, raw)

    simplified = step2.simplify(0.000005)
    step3 = simplified if (not step2.is_valid or simplified.is_valid) else step2
    report("2. After simplify (0.000005)", step3, step2, raw)

    step4 = set_precision(step3, 0.000001, mode="pointwise")
    report("3. After set_precision (1e-6)", step4, step3, raw)

    print("\n--- Effect of varying precision ---")

    for dp in [6, 7, 8]:
        test = shapely.wkt.loads(dump_wkt(raw, precision=dp))
        test = test.simplify(0.000005)
        test = set_precision(test, 10**-dp, mode="pointwise")
        try:
            coords = len(list(test.geoms[0].exterior.coords))
        except AttributeError:
            coords = len(list(test.exterior.coords))
        diff = test.symmetric_difference(raw).area * 111000 * 70000
        print(f"  {dp}dp: vertices={coords}, diff vs raw={diff:.4f} m²")

    print("\n--- Effect of removing simplification ---")
    test_no_simplify = shapely.wkt.loads(dump_wkt(raw))
    test_no_simplify = set_precision(test_no_simplify, 0.000001, mode="pointwise")
    try:
        coords = len(list(test_no_simplify.geoms[0].exterior.coords))
    except AttributeError:
        coords = len(list(test_no_simplify.exterior.coords))
    diff = test_no_simplify.symmetric_difference(raw).area * 111000 * 70000
    print(f"  No simplify: vertices={coords}, diff vs raw={diff:.4f} m²")


In [58]:
check_geometry(barnet_data, "Totteridge")


--- Checking geometry for 'Totteridge' ---
0. Raw                              vertices=2971  vs_raw=-  vs_prev=-
1. After applying 6 d.p.            vertices=2971  vs_raw=224.4193 m²  vs_prev=224.4193 m²
2. After simplify (0.000005)        vertices= 520  vs_raw=1010.3070 m²  vs_prev=969.9118 m²
3. After set_precision (1e-6)       vertices= 520  vs_raw=857.9129 m²  vs_prev=0.0000 m²

--- Effect of varying precision ---
  6dp: vertices=520, diff vs raw=857.9129 m²
  7dp: vertices=514, diff vs raw=1006.6178 m²
  8dp: vertices=515, diff vs raw=1008.6967 m²

--- Effect of removing simplification ---
  No simplify: vertices=2971, diff vs raw=0.1166 m²


The simplification step is almost entirely responsible:

| Step | Vertices | vs Raw |
|------|----------|--------|
| Raw | 2971 | - |
| 6dp round-trip | 2971 | 224 m² |
| simplify(0.000005) | 520 | 1010 m² |
| set_precision(1e-6) | 520 | 858 m² |


Simplification removes 82% of vertices (2971 → 520) and causes ~1000 m² of change. Without simplification, the only difference from raw is 0.12 m² - essentially negligible rounding.

Interestingly, increasing decimal places (6 -> 7 -> 8) made the precision worse. Therefore 6 d.p. is acting as a pre-smoothing step.


---
Check other CA geometries to make sure they have the same effects

In [59]:
check_geometry(barnet_data, "Monken Hadley")


--- Checking geometry for 'Monken Hadley' ---
0. Raw                              vertices=2192  vs_raw=-  vs_prev=-
1. After applying 6 d.p.            vertices=2192  vs_raw=282.4037 m²  vs_prev=282.4037 m²
2. After simplify (0.000005)        vertices= 468  vs_raw=1390.6041 m²  vs_prev=1344.0054 m²
3. After set_precision (1e-6)       vertices= 468  vs_raw=1200.3912 m²  vs_prev=0.0000 m²

--- Effect of varying precision ---
  6dp: vertices=468, diff vs raw=1200.3912 m²
  7dp: vertices=476, diff vs raw=1317.8558 m²
  8dp: vertices=477, diff vs raw=1310.4371 m²

--- Effect of removing simplification ---
  No simplify: vertices=2192, diff vs raw=1.1072 m²


In [60]:
check_geometry(barnet_data, "Wood Street")


--- Checking geometry for 'Wood Street' ---
0. Raw                              vertices=1196  vs_raw=-  vs_prev=-
1. After applying 6 d.p.            vertices=1196  vs_raw=89.1774 m²  vs_prev=89.1774 m²
2. After simplify (0.000005)        vertices= 271  vs_raw=365.7166 m²  vs_prev=348.5242 m²
3. After set_precision (1e-6)       vertices= 271  vs_raw=298.2786 m²  vs_prev=0.0000 m²

--- Effect of varying precision ---
  6dp: vertices=271, diff vs raw=298.2786 m²
  7dp: vertices=266, diff vs raw=348.6700 m²
  8dp: vertices=261, diff vs raw=387.0178 m²

--- Effect of removing simplification ---
  No simplify: vertices=1196, diff vs raw=0.1865 m²
